In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 64     # how many independent sequences we process in parallel, B
block_size = 256       # maximum context length for predictions, T
n_embd = 384 # embedding vector size, C
n_head = 6
n_layer = 6
dropout = 0.2
max_iters = 500
eval_interval = 500
learning_rate = 3e-4
eval_iters = 200
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
# ------------

torch.manual_seed(1337)

with open('nanogpt/input/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
  return [stoi[c] for c in s]

def decode(l):
  return ''.join([itos[i] for i in l.tolist()])

In [3]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

xb, yb = get_batch('train')
print(decode(xb[0]))
print(xb.shape, yb.shape)
print(xb.device, yb.device)

 royalties and rights of banish'd Hereford?
Is not Gaunt dead, and doth not Hereford live?
Was not Gaunt just, and is not Harry true?
Did not the one deserve to have an heir?
Is not his heir a well-deserving son?
Take Hereford's rights away, and take from 
torch.Size([64, 256]) torch.Size([64, 256])
mps:0 mps:0


In [ ]:


class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.head_size = head_size
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) # not optimized but part of Head

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x) # (B,T, head_size)
        q = self.query(x) # B,T, head_size
        wei = q @ k.transpose(-2, -1) * self.head_size**-0.5 # B, T, T

        tril = self.tril[:T, :T]

        wei = wei.masked_fill(tril == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x) #B,T, head_size

        out = wei @ v # B, T, head_size

        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        outs = []
        for head in self.heads:
            outs.append(head(x))

        outs = torch.cat(outs, dim=-1) # B, T, head_size * num_heads = B, T, C

        return self.dropout(self.proj(outs))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        ) 

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = MultiHeadAttention(n_head, head_size)
        self.ff = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.heads(self.ln1(x))
        x = x + self.ff(self.ln2(x))

        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embeddings = nn.Embedding(vocab_size, n_embd)
        self.pos_embeddings = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size) 


    def forward(self, idx, targets=None):

        B, T = idx.shape 
        tok_emb = self.token_embeddings(idx)
        pos_emb = self.pos_embeddings(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx[:, -block_size:])
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx



In [5]:
xb.shape

torch.Size([64, 256])

In [6]:
model = GPTLanguageModel()
model.to(device)
print(sum(p.numel() for p in model.parameters()))
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for steps in range(max_iters):
  xb, yb = get_batch('train')

  logits, loss = model(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

  if (steps % eval_interval == 0) or (steps == max_iters - 1):
    losses = estimate_loss()
    print(losses['train'], losses['val'])


10788929


KeyboardInterrupt: 

In [ ]:
print(decode(model.generate(idx = torch.zeros((1, 1), dtype=torch.long).to(device), max_new_tokens=500)[0]))


Quse have gor thee the in ite muemee, am yould who an;
Soundains thins, a hat sircentirds?
Then, his to mach faintietis fetherssh,
Thath his olligr istiveVinll wo'ty.

Play:
May, ithe the of lorke commean her sanown
Tonger, of is orsut witnster wers,
swould ank'd rovis inge hifaled?
Rom, haves thould dus mall sand nognd not not 'to shanses;
And tely, plest weasthoust heret t,
Aurelnght nous shat, Tho weas drashethin
A dotht; herd the a day air, of the ince jonce;
Onand shiny; be wo wine a gehian
